In [81]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')

# Load Dataset
data = pd.read_csv('train.txt', sep=';', header=None, names=['text', 'emotion'])
df = data.copy()

df.head()

# Check Missing Values
df.isna().sum()

# Convert Emotion into Numbers
unique_emotions = df['emotion'].unique()

emotion_numbers = {}
i = 0

for emo in unique_emotions:
    emotion_numbers[emo] = i
    i += 1

df['emotion'] = df['emotion'].map(emotion_numbers)

df.head()

# Convert into Lowercase
df['text'] = df['text'].apply(lambda x: x.lower())

df.head()

# Remove Punctuation
def remove_punc(txt):
    return txt.translate(str.maketrans('', '', string.punctuation))

df['text'] = df['text'].apply(remove_punc)

# Remove Numbers
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new += i
    return new

df['text'] = df['text'].apply(remove_numbers)

# Remove Emojis / Non-ASCII Characters
def remove_emojies(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojies)

# Stop Words
stop_words = set(stopwords.words('english'))

# Remove Stop Words
def remove(txt):
    words = txt.split()
    cleaned = []

    for i in words:
        if i not in stop_words:
            cleaned.append(i)

    return " ".join(cleaned)

df['text'] = df['text'].apply(remove)

df.head()

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['emotion'],
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_train.head())

# -----------------------------
# Bag of Words
# -----------------------------
bow_vectorizer = CountVectorizer()

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# Naive Bayes
nb_model = MultinomialNB()

nb_model.fit(X_train_bow, y_train)

pred_nb = nb_model.predict(X_test_bow)

print("Bag of Words Accuracy :", accuracy_score(y_test, pred_nb))

# -----------------------------
# TF-IDF
# -----------------------------
tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Naive Bayes
nb_model2 = MultinomialNB()

nb_model2.fit(X_train_tfidf, y_train)

pred_nb2 = nb_model2.predict(X_test_tfidf)

print("TF-IDF + Naive Bayes Accuracy :", accuracy_score(y_test, pred_nb2))

# Logistic Regression
model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("TF-IDF + Logistic Regression Accuracy :", accuracy_score(y_test, y_pred))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


(12800,)
676      refers course though cant help feeling somehow...
12113                im starting feel im suffering fatigue
7077     feel like probably would liked book little bit...
13005                                  really feel awkward
12123    im feeling little grumpy today lame weather te...
Name: text, dtype: object
Bag of Words Accuracy : 0.768125
TF-IDF + Naive Bayes Accuracy : 0.6609375
TF-IDF + Logistic Regression Accuracy : 0.8628125
